<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1:
- Nombre de alumno 2:


Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/...../)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [75]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv('sales.csv', parse_dates=["date"])
df.head()

C:\Users\admin\AppData\Local\Temp\ipykernel_24444\299739943.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('sales.csv', parse_dates=["date"])


,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
0,0,2012-01-31,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,glass,500ml,0.96,13280
1,1,2012-01-31,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,plastic,1.5lt,2.86,6727
2,2,2012-01-31,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,can,330ml,0.87,9848
3,3,2012-01-31,Athens,37.97945,23.71622,672130,shop_1,adult-cola,glass,500ml,1.00,20050
4,4,2012-01-31,Athens,37.97945,23.71622,672130,shop_1,adult-cola,can,330ml,0.39,25696


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

5. El mae, es la diferencia absoluta entre las predicciones del modelo y cantidad real. Es decir en este caso, con los 13542, de MAE dummy tenemos un error de esta cantidad.
6. Al hacerlo con el xgboost podemos ver una mejora clara en que el error se reduce a 2505 unidades. Es decir redujo el error en casi 11000 unidades. Un modelo que se acerco mucho mas al promedio real.

In [76]:
from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

set_config(transform_output="pandas")


X = df.drop(columns=['quantity'])
y = df['quantity']

# División train/val/test: 70/20/10
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42
)


# Ahora split train/val de lo restante en 70/20 respecto al total 
fraccion_of_trainval = 20 / (70 + 20) 
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=fraccion_of_trainval, random_state=42
)

def transformadorFecha(df):
    df = df.copy()
    df["year"] = pd.to_datetime(df["date"]).dt.year.astype(int)
    df["month"] = pd.to_datetime(df["date"]).dt.month.astype(int)
    df["day"] = pd.to_datetime(df["date"]).dt.day.astype(int)
    return df.drop(columns=["date"])


tran_Fecha = FunctionTransformer(transformadorFecha)


print("Tamaños: ", X_train.shape, X_val.shape, X_test.shape)

Tamaños:  (5218, 11) (1492, 11) (746, 11)


In [77]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object","category"]).columns.tolist()


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ],
    remainder="passthrough"
).set_output(transform="pandas")


In [78]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor

dummy_pipeline = Pipeline(steps=[
    ("date_features", tran_Fecha),
    ("preprocessor", preprocessor),
    ("regressor", DummyRegressor(strategy="mean"))
])

In [79]:
from sklearn.metrics import mean_absolute_error

dummy_pipeline.fit(X_train, y_train)
y_val_pred = dummy_pipeline.predict(X_val)

mae_dummy = mean_absolute_error(y_val, y_val_pred)
print("MAE Dummy:", mae_dummy)


MAE Dummy: 13543.961387782238


In [80]:
from xgboost import XGBRegressor

xgb_pipeline = Pipeline(steps=[
    ("date_features", tran_Fecha),
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(random_state=42))
])

xgb_pipeline.fit(X_train, y_train)
y_val_pred_xgb = xgb_pipeline.predict(X_val)

mae_xgb = mean_absolute_error(y_val, y_val_pred_xgb)
print("MAE XGB:", mae_xgb)


MAE XGB: 2505.027099609375


In [81]:
import joblib

joblib.dump(dummy_pipeline, "dummy_model.pkl")
joblib.dump(xgb_pipeline, "xgb_model.pkl")


['xgb_model.pkl']

## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

1. El punto de usar una restriccion monotona negativa es obligar al modelo a aprender la relacion economica, si el precio sube la cantidad vendida no deberia aumentar, haciendo mas realista el modelo.

3. El error bajo a 2389, por lo que el amigo tenia razon, hay una relacion inversamente proporcional entre el precio y la demanda, sin embargo, hay que destacar que no es tan significativa ya que solo ser redujo en alrededor de 150.

In [82]:
from xgboost import XGBRegressor
import numpy as np

# Ajustamos pipeline
prep_only = Pipeline(steps=[
    ("date_features", tran_Fecha),
    ("preprocessor", preprocessor)
])

preprocessor.fit(X_train)

# Obtenemos las columnas transformadas
feature_names = preprocessor.get_feature_names_out()


# Definimos restricciones: 0 para todas, -1 sólo en 'price'
constraints = [0] * len(feature_names)


for i, col in enumerate(feature_names):
    if "price" in col: 
        constraints[i] = -1

print("Constraints asignados:", constraints)

Constraints asignados: [0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [83]:
# Creamos el modelo con restricciones
xgb_mono = XGBRegressor(
    random_state=42,
    monotone_constraints=tuple(constraints),
    tree_method="hist",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1
)

# Armamos el pipeline completo
xgb_pipeline_mono = Pipeline(steps=[
    ("date_features", tran_Fecha),
    ("preprocessor", preprocessor),
    ("regressor", xgb_mono)
])

# Entrenamos
xgb_pipeline_mono.fit(X_train, y_train)



,steps,"[('date_features', ...), ('preprocessor', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function tra...00238B0B80180>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [84]:
# Validación
y_val_pred_mono = xgb_pipeline_mono.predict(X_val)
mae_xgb_mono = mean_absolute_error(y_val, y_val_pred_mono)
print("MAE XGB con monotonicidad:", mae_xgb_mono)

MAE XGB con monotonicidad: 2389.55859375


In [85]:
joblib.dump(xgb_pipeline_mono, "xgb_model_monotonic.pkl")
print("Modelo guardado como xgb_model_monotonic.pkl")

Modelo guardado como xgb_model_monotonic.pkl


## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [86]:
import optuna
from optuna.samplers import TPESampler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import joblib

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [87]:
def objective(trial):
    # Hyperparametros
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1, log=True)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 1.0)
    min_freq = trial.suggest_float("min_frequency", 0.0, 1.0)

    # HiperparametroOneHotEncoder
    min_freq = trial.suggest_float("min_frequency", 0.0, 1.0)
    preprocessor.set_params(cat__min_frequency=min_freq)

    # Modelo XGBoost
    xgb = XGBRegressor(
        random_state=42,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        n_jobs=-1,
        tree_method="hist"
    )

    # Pipeline
    pipe = Pipeline(steps=[
        ("date_features", tran_Fecha),
        ("preprocessor", preprocessor),
        ("regressor", xgb)
    ])

    # Fit y evaluar
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)

    # Guardar pipeline en el trial
    trial.set_user_attr("pipeline", pipe)

    return mae

In [88]:
# Optimizacion
study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study.optimize(objective, timeout=300)

# Resultados
print("Número de trials:", len(study.trials))
print("Mejor MAE:", study.best_value)
print("Mejores hiperparámetros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# Mejor Pipeline
best_pipeline = study.best_trial.user_attrs["pipeline"]


# Constraints de 'price'
preprocessor.fit(X_train)
feature_names = preprocessor.get_feature_names_out()
constraints = [0] * len(feature_names)
for i, col in enumerate(feature_names):
    if "price" in col:
        constraints[i] = -1

xgb_final = XGBRegressor(
    random_state=42,
    **study.best_params,
    monotone_constraints=str(tuple(constraints)),
    n_jobs=-1,
    tree_method="hist"
)

modelo_final = Pipeline(steps=[
    ("date_features", tran_Fecha),
    ("preprocessor", preprocessor),
    ("regressor", xgb_final)
])

modelo_final.fit(X_train, y_train)

# Guardar modelo
joblib.dump(modelo_final, "best_xgb_optuna.pkl")
print("Modelo final guardado como 'best_xgb_optuna.pkl'")

Número de trials: 316
Mejor MAE: 2138.283203125
Mejores hiperparámetros:
  learning_rate: 0.034417199585665775
  n_estimators: 876
  max_depth: 9
  max_leaves: 92
  min_child_weight: 3
  reg_alpha: 0.38495010745539826
  reg_lambda: 0.22124200542244415
  min_frequency: 0.07953151844268458


c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:49:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "min_frequency" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Modelo final guardado como 'best_xgb_optuna.pkl'


## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [89]:
# Inserte su código acá

## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [90]:
# Inserte su código acá

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [91]:
# Inserte su código acá

# Conclusión
Exito!
<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>